<a href="https://colab.research.google.com/github/ibmm-unibe-ch/FrankenMSA/blob/ngrok/app/FrankenMSA_app_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://github.com/ibmm-unibe-ch/FrankenMSA/blob/dev/app/assets/frankenmsa_header.png?raw=true)

# [FrankenMSA](https://github.com/ibmm-unibe-ch/FrankenMSA/tree/main/)
This notebook can run the [FrankenMSA App](https://github.com/ibmm-unibe-ch/FrankenMSA/tree/main/) to provide a graphical user interface for manipulating Multiple Sequence Alignments.

We highly recommend to use an external window for best experience. Firefox users might need to use the [ngrok](https://ngrok.com/) option. Otherwise, the inline option might be used as a last option.

Press the "Runtime" --> "Run all" once you made your selection or press "Ctrl + F9".

In [1]:
WindowMode = 'ngrok' # @param ["external", "inline", "ngrok"] {allow-input: true}


# Code

In [ ]:
# === Cell 1 (robust): clean clone + make bridge ===
import os, sys, shutil, importlib
from pathlib import Path

REPO_URL = "https://github.com/ibmm-unibe-ch/FrankenMSA.git"
BRANCH   = "feature/colab-runner"  


%cd /content

if Path("/content/FrankenMSA").exists():
    print("🧹 Detected existing /content/FrankenMSA, removing old copy…")
    shutil.rmtree("/content/FrankenMSA", ignore_errors=True)


!git clone -q -b {BRANCH} --single-branch {REPO_URL} /content/FrankenMSA


assert os.path.isfile("/content/FrankenMSA/app/proteinmpnn_runner.py"), \
    "proteinmpnn_runner.py 不在该分支，请确认 BRANCH 设置正确（feature/colab-runner 或 main）。"


%pip -q install biopython==1.83 einops==0.7.0 pyngrok


bridge = """
import sys, os
sys.path.append('/content/FrankenMSA/app')
from proteinmpnn_runner import parse_param_string, run_proteinmpnn
"""
with open("/content/colab_bridge.py", "w") as f:
    f.write(bridge)

import colab_bridge  # noqa: F401
print("✅ colab_bridge ready.")

In [ ]:
# === Cell 2 — Start FrankenMSA via ngrok (clean minimal version) ===
import os, sys, re, time, subprocess, getpass
from pathlib import Path

# Clean existing ngrok tunnels
try:
    from pyngrok import ngrok
    for t in ngrok.get_tunnels():
        try: ngrok.disconnect(t.public_url)
        except: pass
    ngrok.kill()
except Exception:
    pass

ROOT = "/content/FrankenMSA"
APP_DIR = f"{ROOT}/app"
assert os.path.isdir(ROOT), "FrankenMSA repo not found. Run Cell 1 first."

print("📦 Installing dependencies...")
req = Path(ROOT) / "requirements.txt"
try:
    if req.is_file():
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "dash==2.16.1", "dash-bootstrap-components==1.5.0",
                               "plotly==5.24.1", "dash-bio==1.0.2"])
except Exception as e:
    print("⚠️ pip warning:", e)

# Ngrok setup
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyngrok"])
from pyngrok import ngrok, conf
token = getpass.getpass("Enter ngrok authtoken (hidden): ").strip().strip("'").strip('"')
conf.get_default().auth_token = token
PORT = 8050
public_url = ngrok.connect(addr=f"0.0.0.0:{PORT}", proto="http").public_url
print("🌐", public_url)

# Patch prevent_initial_call
patch_count = 0
for p in Path(APP_DIR).rglob("*.py"):
    s = p.read_text(encoding="utf-8", errors="ignore")
    s2, n1 = re.subn(r",\s*\{\s*['\"]prevent_initial_call['\"]\s*:\s*True\s*\}", ", prevent_initial_call=True", s)
    s3, n2 = re.subn(r"clientside_callback\((.*?)\s*,\s*\{\s*['\"]prevent_initial_call['\"]\s*:\s*True\s*\}\s*\)",
                     r"clientside_callback(\1, prevent_initial_call=True)", s2, flags=re.DOTALL)
    if n1 or n2:
        p.write_text(s3, encoding="utf-8")
        patch_count += n1 + n2
print(f"🩹 Patched {patch_count} file(s)")

# Launch app
env = os.environ.copy()
env["PORT"], env["HOST"] = str(PORT), "0.0.0.0"
proc = subprocess.Popen([sys.executable, "app/app.py"], cwd=ROOT, env=env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

# Wait for server to start
start = time.time()
while time.time() - start < 20:
    line = proc.stdout.readline()
    if not line:
        time.sleep(0.2); continue
    if any(k in line for k in ["Running on", "Dash is running", "Listening at"]):
        break

print("🚀 App running at:", public_url)